# 환경변수 불러오기

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

# STT(Speech to Text)

In [5]:
# pyaudio speechrecognition pydub
import speech_recognition as sr

r = sr.Recognizer()
# 실습 해보면서 넣을 수 있는 옵션 구글링해보기
with sr.Microphone() as source:
    print("말해주세요: ")
    r.adjust_for_ambient_noise(source, duration=1) # 마이크 주변 환경 소음을 자동으로 조정
    audio = r.listen(source) # 마이크 입력받기
    print("인식 중입니다.....")
    text = r.recognize_openai(audio) # 텍스트로 변환
    print(f"인식된 텍스트: {text}")

    audio_file = audio.get_wav_data() # 음성파일 추출
    with open("audio/input.wav", "wb") as f: # 음성파일 저장
        f.write(audio_file)
    print("목소리 저장 완료")


말해주세요: 
인식 중입니다.....
인식된 텍스트: Bye. 
목소리 저장 완료


In [ ]:
# 오디오 출력하기
from pydub import AudioSegment
from pydub.playback import play
sound = AudioSegment.from_wav("audio/input.wav")
play(sound)

# LLM 연결하기

In [7]:
from openai import OpenAI

client = OpenAI()

def chat(user_text):
    system_prompt = """ 

    """

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ]
    )

    return response.choices[0].message.content

# 챗봇과 대화해보기

In [33]:
while True:
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요: ")
        r.adjust_for_ambient_noise(source, duration=1) # 마이크 주변 환경 소음을 자동으로 조정
        audio = r.listen(source) # 마이크 입력받기
        print("인식 중입니다.....")
        user_text = r.recognize_openai(audio) # 텍스트로 변환
        print(f"인식된 텍스트: {user_text}")

        if user_text == "이제 그만하자":
            break
        answer = chat(user_text)
        print(f"챗봇 답변: {answer}")


말해주세요: 
인식 중입니다.....
인식된 텍스트: 이제 그만하자


# TTS(Text to Speech)

In [1]:
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="nova",
    input="배고파요",
    instructions="낮고 차분한 목소리로 말해줘"
) as response:
    response.stream_to_file("audio/speech.mp3")

In [2]:
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("audio/speech.mp3")
play(sound)

In [8]:
import tempfile

while True:
    r = sr.Recognizer()

    with sr.Microphone() as source:
        print("듣는 중...")
        # STEP1 마이크로부터 입력
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)
        print("인식 중입니다....")

        #STEP2. Whisper API 를 통한 텍스트 변환
        user_text = r.recognize_openai(audio)
        print(f"인식된 텍스트: {user_text}")

        if user_text == "그만":
            break
        # STEP3. 인공지능 챗봇 응답
        answer = chat(user_text)
        print(f"챗봇 답변: {answer}")

        # STEP4. Whisper API로 응답
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input="answer",
            instructions="밝은 목소리로 말해줘"
        ) as response:
            #음성 합성 결과를 임시 파일로 저장
            with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as temp_file:
                temp_path = temp_file.name
                response.stream_to_file(temp_path)

                #재생
                sound = AudioSegment.from_mp3(temp_path)
                play(sound)

듣는 중...
인식 중입니다....
인식된 텍스트: Uh, uh, uh, uh.
챗봇 답변: Hello! How can I assist you today?
듣는 중...
인식 중입니다....
인식된 텍스트: 그만
